# DriveSense-VLM — 04: Full 4-Level Evaluation

**GPU**: A100 80GB | **Time**: ~1 h | **Cost**: ~12 CU

- **Level 1** — Grounding accuracy (IoU + Hungarian matching)
- **Level 2** — Reasoning quality (LLM-as-judge via Claude API)
- **Level 3** — Production benchmarks (reads `03_benchmark.ipynb` output)
- **Level 4** — Robustness (stratified by time-of-day, weather, OOD)

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **Level 2**: Store `ANTHROPIC_API_KEY` in Colab Secrets, or set `USE_MOCK_JUDGE=True`.
>
> **Prerequisites**: `01_training.ipynb` (checkpoint) + `03_benchmark.ipynb` (JSONs).

In [ ]:
# ══════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════
USE_MOCK_JUDGE = True   # True: free random scores; False: real Claude API judge
# ══════════════════════════════════════════════════════════

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD (skip if already cloned — saves bandwidth)
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Install evaluation dependencies
# Note: nuscenes-devkit (used in nb00/05) must be installed with --no-deps
#       to skip its strict numpy version pin. Not needed in this notebook.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install pyyaml tqdm Pillow requests -q 2>&1 | tail -1
!pip install transformers peft accelerate -q 2>&1 | tail -1
!pip install scikit-learn scipy -q 2>&1 | tail -1

import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → A100 GPU"
print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import drivesense
print("✓ DriveSense loaded")

In [ ]:
# Generate predictions on the test set (always regenerates)
import os, glob

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"
PRED_DIR  = f"{OUTPUTS_ROOT}/predictions"
PRED_FILE = f"{PRED_DIR}/test_predictions.jsonl"
os.makedirs(PRED_DIR, exist_ok=True)

best = f"{TRAIN_OUT}/checkpoint-best"
if not os.path.exists(best):
    ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
    best = ckpts[-1] if ckpts else None

os.chdir(REPO_ROOT)
if best and os.path.exists(best):
    print(f"Generating predictions from {best}...")
    !python scripts/run_generate_predictions.py \
        --split test \
        --adapter-path {best} \
        --output {PRED_FILE} \
        2>&1
else:
    print("No checkpoint found — using mock predictions.")
    !python scripts/run_generate_predictions.py \
        --split test \
        --output {PRED_FILE} \
        --mock \
        2>&1

with open(PRED_FILE) as f:
    n = sum(1 for _ in f)
print(f"✓ {n} predictions → {PRED_FILE}")

In [ ]:
# Level 1 — Grounding accuracy (IoU + Hungarian matching)
import os, json

PRED_FILE = f"{OUTPUTS_ROOT}/predictions/test_predictions.jsonl"
EVAL_L1   = f"{OUTPUTS_ROOT}/eval/level1"
os.makedirs(EVAL_L1, exist_ok=True)

os.chdir(REPO_ROOT)
!python scripts/run_evaluation.py \
    --level 1 \
    --predictions {PRED_FILE} \
    --output-dir {EVAL_L1} \
    2>&1

results = f"{EVAL_L1}/grounding_metrics.json"
if os.path.exists(results):
    m = json.loads(open(results).read())
    print("\nLevel 1 — Grounding Metrics:")
    for k, v in m.items():
        print(f"  {k}: {v}")

In [ ]:
# Level 2 — Reasoning quality (LLM-as-judge)
import os, json

PRED_FILE = f"{OUTPUTS_ROOT}/predictions/test_predictions.jsonl"
EVAL_L2   = f"{OUTPUTS_ROOT}/eval/level2"
os.makedirs(EVAL_L2, exist_ok=True)

if not USE_MOCK_JUDGE:
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        print("✓ ANTHROPIC_API_KEY loaded from Colab Secrets")
    except Exception:
        print("⚠ Could not load API key — falling back to mock judge.")
        USE_MOCK_JUDGE = True

mock_flag = "--mock-judge" if USE_MOCK_JUDGE else ""
if USE_MOCK_JUDGE:
    print("Using mock judge (random scores in [3,5]).")

os.chdir(REPO_ROOT)
!python scripts/run_evaluation.py \
    --level 2 \
    --predictions {PRED_FILE} \
    --output-dir {EVAL_L2} \
    {mock_flag} \
    2>&1

results = f"{EVAL_L2}/reasoning_metrics.json"
if os.path.exists(results):
    m = json.loads(open(results).read())
    print("\nLevel 2 — Reasoning Metrics:")
    for k, v in m.items():
        print(f"  {k}: {v}")

In [ ]:
# Level 3 (production) + Level 4 (robustness)
import os, json

PRED_FILE = f"{OUTPUTS_ROOT}/predictions/test_predictions.jsonl"
BENCH_DIR = f"{OUTPUTS_ROOT}/benchmarks"
EVAL_L3   = f"{OUTPUTS_ROOT}/eval/level3"
EVAL_L4   = f"{OUTPUTS_ROOT}/eval/level4"
os.makedirs(EVAL_L3, exist_ok=True)
os.makedirs(EVAL_L4, exist_ok=True)

os.chdir(REPO_ROOT)
!python scripts/run_evaluation.py \
    --level 3 \
    --benchmarks-dir {BENCH_DIR} \
    --output-dir {EVAL_L3} \
    2>&1

!python scripts/run_evaluation.py \
    --level 4 \
    --predictions {PRED_FILE} \
    --output-dir {EVAL_L4} \
    2>&1

for level, path in [("Level 3", f"{EVAL_L3}/production_metrics.json"),
                    ("Level 4", f"{EVAL_L4}/robustness_metrics.json")]:
    if os.path.exists(path):
        m = json.loads(open(path).read())
        print(f"\n{level}:")
        for k, v in list(m.items())[:6]:
            print(f"  {k}: {v}")

In [ ]:
import os

EVAL_DIR = f"{OUTPUTS_ROOT}/eval"
os.makedirs(EVAL_DIR, exist_ok=True)

os.chdir(REPO_ROOT)
!python scripts/run_full_evaluation.py \
    --generate-report \
    --output-dir {EVAL_DIR} \
    2>&1

for candidate in [f"{EVAL_DIR}/final_report.txt",
                  f"{EVAL_DIR}/evaluation_report.txt"]:
    if os.path.exists(candidate):
        print(open(candidate).read())
        break

print("\nEvaluation files on Drive:")
for root, dirs, files in os.walk(EVAL_DIR):
    for fname in sorted(files):
        p = os.path.join(root, fname)
        rel = os.path.relpath(p, EVAL_DIR)
        print(f"  {rel}  ({os.path.getsize(p)/1024:.1f} KB)")